In [69]:
import pandas as pd
from pathlib import Path

V12_PATH = Path("../data/raw/Version_8")
V13_PATH = Path("../data/raw/Version_9")

In [70]:
postings_v13=pd.read_csv(V13_PATH / "job_postings.csv")
postings_v12=pd.read_csv(V12_PATH / "job_postings.csv")

companies_v12 = pd.read_csv(V12_PATH / "companies.csv")
companies_v13 = pd.read_csv(V13_PATH / "companies.csv")

Compare Shapes

In [71]:
comparison = pd.DataFrame({
    "Version 12":[
        postings_v12.shape,
        companies_v12.shape
    ],
    "Version 13":[
        postings_v13.shape,
        companies_v13.shape
    ]
},
index=[
    "postings",
    "companies"
])

comparison

,Version 12,Version 13
postings,"(15886, 27)","(33246, 28)"
companies,"(6063, 10)","(11361, 10)"


Compare Columns

In [72]:
v12_cols = set(postings_v12.columns)
v13_cols = set(postings_v13.columns)

print("Added in Version 13")
print(sorted(v13_cols - v12_cols))

print()

print("Removed in Version 13")
print(sorted(v12_cols - v13_cols))

Added in Version 13
['scraped']

Removed in Version 13
[]


Compare Data Types

In [73]:
pd.DataFrame({
    "V12": postings_v12.dtypes,
    "V13": postings_v13.dtypes
})

,V12,V13
application_type,object,object
application_url,object,object
applies,float64,float64
closed_time,float64,float64
company_id,float64,float64
compensation_type,object,object
currency,object,object
description,object,object
expiry,int64,float64
formatted_experience_level,object,object


Compare Missing Values 

In [74]:
missing = pd.DataFrame({
    "Version 12": postings_v12.isna().sum(),
    "Version 13": postings_v13.isna().sum()
})

missing

,Version 12,Version 13
application_type,0.0,0
application_url,6091.0,12250
applies,7186.0,17008
closed_time,14958.0,32074
company_id,366.0,654
compensation_type,9384.0,19894
currency,9384.0,19894
description,1.0,1
expiry,0.0,0
formatted_experience_level,4902.0,9181


Compare Date Ranges

In [75]:
for name, df in [
    ("V9", postings_v12),
    ("V13", postings_v13)
]:

    dates = pd.to_datetime(
        df["original_listed_time"],
        unit="ms"
    )

    print(name)
    print("Rows:", len(df))
    print("Earliest:", dates.min())
    print("Latest:", dates.max())
    print()

V9
Rows: 15886
Earliest: 2023-06-08 07:49:05
Latest: 2023-08-24 09:13:30

V13
Rows: 33246
Earliest: 2023-06-08 07:40:00
Latest: 2023-11-04 09:26:40



Compare Duplicate Keys

In [76]:
print(postings_v12["job_id"].duplicated().sum())

print(postings_v13["job_id"].duplicated().sum())

0
0


Compare Unique Jobs

In [77]:
print(postings_v12["job_id"].nunique())

print(postings_v13["job_id"].nunique())

15886
33246


Compare Common Job IDs 

In [78]:
jobs_v12 = set(postings_v12["job_id"])
jobs_v13 = set(postings_v13["job_id"])

print("Common Jobs:", len(jobs_v12 & jobs_v13))
print("Only in V9:", len(jobs_v12 - jobs_v13))
print("Only in V13:", len(jobs_v13 - jobs_v12))

Common Jobs: 15886
Only in V9: 0
Only in V13: 17360


In [81]:
import pandas as pd
from pathlib import Path

# ==========================
# CHANGE ONLY THESE
# ==========================
VERSION_A = "Version_9"
VERSION_B = "Version_13"

DATA_PATH = Path("../data/raw")

# Load postings
postings_A = pd.read_csv(DATA_PATH / VERSION_A / "job_postings.csv")
postings_B = pd.read_csv(DATA_PATH / VERSION_B / "postings.csv")

# Convert job_ids to sets
jobs_A = set(postings_A["job_id"])
jobs_B = set(postings_B["job_id"])

# Compare
common_jobs = jobs_A & jobs_B
only_A = jobs_A - jobs_B
only_B = jobs_B - jobs_A

print("="*60)
print(f"Comparing {VERSION_A} vs {VERSION_B}")
print("="*60)

print(f"Total Jobs in {VERSION_A}: {len(jobs_A):,}")
print(f"Total Jobs in {VERSION_B}: {len(jobs_B):,}\n")

print(f"Common Jobs           : {len(common_jobs):,}")
print(f"Only in {VERSION_A}    : {len(only_A):,}")
print(f"Only in {VERSION_B}    : {len(only_B):,}")

# Percentage overlap
print("\nOverlap Percentage")
print(f"{VERSION_A} contained in {VERSION_B}: {len(common_jobs)/len(jobs_A)*100:.2f}%")
print(f"{VERSION_B} contained in {VERSION_A}: {len(common_jobs)/len(jobs_B)*100:.2f}%")

dates_A = pd.to_datetime(postings_A["original_listed_time"], unit="ms")
dates_B = pd.to_datetime(postings_B["original_listed_time"], unit="ms")

print("\nTimeline")
print(f"{VERSION_A}: {dates_A.min()}  -->  {dates_A.max()}")
print(f"{VERSION_B}: {dates_B.min()}  -->  {dates_B.max()}")

Comparing Version_9 vs Version_13
Total Jobs in Version_9: 33,246
Total Jobs in Version_13: 123,849

Common Jobs           : 0
Only in Version_9    : 33,246
Only in Version_13    : 123,849

Overlap Percentage
Version_9 contained in Version_13: 0.00%
Version_13 contained in Version_9: 0.00%

Timeline
Version_9: 2023-06-08 07:40:00  -->  2023-11-04 09:26:40
Version_13: 2023-12-05 21:08:53  -->  2024-04-20 00:26:43


In [82]:
v9_cols = set(postings_A.columns)
v13_cols = set(postings_B.columns)

print("Columns only in Version 9:")
print(sorted(v9_cols - v13_cols))

print("\nColumns only in Version 13:")
print(sorted(v13_cols - v9_cols))

print("\nCommon columns:")
print(len(v9_cols & v13_cols))

Columns only in Version 9:
['scraped']

Columns only in Version 13:
['company_name', 'fips', 'normalized_salary', 'zip_code']

Common columns:
27


In [83]:
schema = pd.DataFrame({
    "Version 9": postings_A.dtypes,
    "Version 13": postings_B.dtypes
})

schema

,Version 9,Version 13
application_type,object,object
application_url,object,object
applies,float64,float64
closed_time,float64,float64
company_id,float64,float64
company_name,NaN,object
compensation_type,object,object
currency,object,object
description,object,object
expiry,float64,float64


In [84]:
missing = pd.DataFrame({
    "V9 Missing %":
        postings_A.isna().mean()*100,

    "V13 Missing %":
        postings_B.isna().mean()*100
})

missing.sort_values(
    "V13 Missing %",
    ascending=False
)

,V9 Missing %,V13 Missing %
closed_time,96.474764,99.133622
skills_desc,98.986344,98.030666
med_salary,93.259339,94.929309
remote_allowed,85.556157,87.689848
applies,51.158034,81.170619
max_salary,66.579438,75.944093
min_salary,66.579438,75.944093
currency,59.838778,70.873402
pay_period,59.838778,70.873402
normalized_salary,NaN,70.873402
